# EDA con PySpark — NOAA Global Surface Summary of the Day (GSOD)

**Objetivo:** abrir y revisar con PySpark el dataset NOAA GSOD.
El notebook descarga años seleccionados desde NOAA, carga los CSV con PySpark, revisa estructura, estadísticas generales, valores faltantes/sentinela y genera resultados relevantes para el tema de investigación sobre cambio climático.

In [1]:
# ============================================================
# 1. Configuración general
# ============================================================

from pathlib import Path
import os
import tarfile
import urllib.request
import shutil
import sys

YEARS = range(2020, 2026)

BASE_URL = "https://www.ncei.noaa.gov/data/global-summary-of-the-day/archive/"
DATA_DIR = Path("data/noaa_gsod")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta de datos:", DATA_DIR.resolve())
print("Años a usar:", list(YEARS))

Carpeta de datos: /Users/luisedgarpoblanodiaz/Downloads/data/noaa_gsod
Años a usar: [2020, 2021, 2022, 2023, 2024, 2025]


In [2]:
# ============================================================
# 2. Descarga y descompresión de datos NOAA GSOD
# ============================================================

def download_and_extract_year(year: int, force_download: bool = False) -> None:
    """Descarga y descomprime un año de NOAA GSOD si no existe localmente."""
    filename = f"{year}.tar.gz"
    url = BASE_URL + filename
    tar_path = DATA_DIR / filename
    extract_dir = DATA_DIR / str(year)

    # Si ya existe la carpeta con CSVs, no descarga de nuevo.
    if extract_dir.exists() and any(extract_dir.rglob("*.csv")) and not force_download:
        print(f"{year}: ya existe localmente, se omite descarga.")
        return

    extract_dir.mkdir(parents=True, exist_ok=True)

    print(f"{year}: descargando {url}")
    urllib.request.urlretrieve(url, tar_path)

    print(f"{year}: descomprimiendo...")
    with tarfile.open(tar_path, "r:gz") as tar:
        try:
            tar.extractall(path=extract_dir, filter="data")
        except TypeError:
            tar.extractall(path=extract_dir)

    tar_path.unlink(missing_ok=True)
    print(f"{year}: listo.")

for year in YEARS:
    download_and_extract_year(year)

csv_count = len(list(DATA_DIR.rglob("*.csv")))
print(f"Total de archivos CSV disponibles: {csv_count:,}")

2020: descargando https://www.ncei.noaa.gov/data/global-summary-of-the-day/archive/2020.tar.gz
2020: descomprimiendo...
2020: listo.
2021: descargando https://www.ncei.noaa.gov/data/global-summary-of-the-day/archive/2021.tar.gz
2021: descomprimiendo...
2021: listo.
2022: descargando https://www.ncei.noaa.gov/data/global-summary-of-the-day/archive/2022.tar.gz
2022: descomprimiendo...
2022: listo.
2023: descargando https://www.ncei.noaa.gov/data/global-summary-of-the-day/archive/2023.tar.gz
2023: descomprimiendo...
2023: listo.
2024: descargando https://www.ncei.noaa.gov/data/global-summary-of-the-day/archive/2024.tar.gz
2024: descomprimiendo...
2024: listo.
2025: descargando https://www.ncei.noaa.gov/data/global-summary-of-the-day/archive/2025.tar.gz
2025: descomprimiendo...
2025: listo.
Total de archivos CSV disponibles: 73,019


In [3]:
# ============================================================
# 3. Verificación del tamaño global descargado
# ============================================================

def folder_size_gb(path: Path) -> float:
    total_bytes = sum(file.stat().st_size for file in path.rglob("*") if file.is_file())
    return total_bytes / (1024 ** 3)

size_gb = folder_size_gb(DATA_DIR)

print(f"Tamaño local descargado: {size_gb:.2f} GB")
print("Cumple mínimo de 1 GB:", "Sí" if size_gb >= 1 else "No")

if size_gb < 1:
    print("Advertencia: descarga más años para cumplir el requisito mínimo de 1 GB.")

Tamaño local descargado: 4.90 GB
Cumple mínimo de 1 GB: Sí


In [7]:
# ============================================================
# 4. Crear SparkSession
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, LongType, FloatType

spark = (
    SparkSession.builder
    .appName("NOAA_GSOD_EDA")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("Python:", sys.executable)

Spark version: 4.0.1
Python: /opt/anaconda3/bin/python


In [8]:
# ============================================================
# 5. Cargar CSVs con PySpark
# ============================================================

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("recursiveFileLookup", True)
    .csv(str(DATA_DIR))
)

df = df_raw
for old_name in df.columns:
    new_name = (
        old_name.strip()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("/", "_")
        .replace("-", "_")
    )
    df = df.withColumnRenamed(old_name, new_name)

df = df.cache()

print("Registros totales:", f"{df.count():,}")
print("Columnas:", len(df.columns))
df.printSchema()
df.show(5, truncate=False)

[Stage 26:=================================================> (2200 + 11) / 2282]

Registros totales: 22,638,932
Columnas: 28
root
 |-- STATION: string (nullable = true)
 |-- DATE: date (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- ELEVATION: double (nullable = true)
 |-- NAME: string (nullable = true)
 |-- TEMP: double (nullable = true)
 |-- TEMP_ATTRIBUTES: double (nullable = true)
 |-- DEWP: double (nullable = true)
 |-- DEWP_ATTRIBUTES: double (nullable = true)
 |-- SLP: double (nullable = true)
 |-- SLP_ATTRIBUTES: double (nullable = true)
 |-- STP: double (nullable = true)
 |-- STP_ATTRIBUTES: double (nullable = true)
 |-- VISIB: double (nullable = true)
 |-- VISIB_ATTRIBUTES: double (nullable = true)
 |-- WDSP: double (nullable = true)
 |-- WDSP_ATTRIBUTES: double (nullable = true)
 |-- MXSPD: double (nullable = true)
 |-- GUST: double (nullable = true)
 |-- MAX: double (nullable = true)
 |-- MAX_ATTRIBUTES: string (nullable = true)
 |-- MIN: double (nullable = true)
 |-- MIN_ATTRIBUTES: string (nullabl

In [9]:
# ============================================================
# 6. Estadística general del dataset
# ============================================================

num_records = df.count()
num_columns = len(df.columns)
station_col = "STATION" if "STATION" in df.columns else None
date_col = "DATE" if "DATE" in df.columns else None

print("=" * 60)
print("RESUMEN GENERAL DEL DATASET")
print("=" * 60)
print(f"Número de registros: {num_records:,}")
print(f"Número de columnas: {num_columns:,}")

if station_col:
    print(f"Estaciones únicas: {df.select(station_col).distinct().count():,}")

if date_col:
    df = df.withColumn("DATE_PARSED", F.to_date(F.col(date_col).cast("string")))
    print("Fecha mínima:", df.agg(F.min("DATE_PARSED")).collect()[0][0])
    print("Fecha máxima:", df.agg(F.max("DATE_PARSED")).collect()[0][0])

print("=" * 60)

RESUMEN GENERAL DEL DATASET
Número de registros: 22,638,932
Número de columnas: 28


Estaciones únicas: 13,008


Fecha mínima: 2020-01-01


[Stage 42:===================================================>(2281 + 1) / 2282]

Fecha máxima: 2025-08-28


In [10]:
# ============================================================
# 7. Tipos de datos por columna
# ============================================================

schema_info = [(field.name, field.dataType.simpleString(), field.nullable) for field in df.schema.fields]
schema_df = spark.createDataFrame(schema_info, ["columna", "tipo_dato", "nullable"])

schema_df.show(100, truncate=False)

+----------------+---------+--------+
|columna         |tipo_dato|nullable|
+----------------+---------+--------+
|STATION         |string   |true    |
|DATE            |date     |true    |
|LATITUDE        |double   |true    |
|LONGITUDE       |double   |true    |
|ELEVATION       |double   |true    |
|NAME            |string   |true    |
|TEMP            |double   |true    |
|TEMP_ATTRIBUTES |double   |true    |
|DEWP            |double   |true    |
|DEWP_ATTRIBUTES |double   |true    |
|SLP             |double   |true    |
|SLP_ATTRIBUTES  |double   |true    |
|STP             |double   |true    |
|STP_ATTRIBUTES  |double   |true    |
|VISIB           |double   |true    |
|VISIB_ATTRIBUTES|double   |true    |
|WDSP            |double   |true    |
|WDSP_ATTRIBUTES |double   |true    |
|MXSPD           |double   |true    |
|GUST            |double   |true    |
|MAX             |double   |true    |
|MAX_ATTRIBUTES  |string   |true    |
|MIN             |double   |true    |
|MIN_ATTRIBU

In [11]:
# ============================================================
# 8. Estadísticas descriptivas para variables numéricas
# ============================================================

numeric_types = {"byte", "short", "int", "bigint", "float", "double", "long"}
numeric_columns = [
    field.name
    for field in df.schema.fields
    if field.dataType.simpleString() in numeric_types
    or field.dataType.simpleString().startswith("decimal")
]

print("Columnas numéricas:", numeric_columns)

if numeric_columns:
    df.select(numeric_columns).describe().show(truncate=False)
else:
    print("No se detectaron columnas numéricas.")

Columnas numéricas: ['LATITUDE', 'LONGITUDE', 'ELEVATION', 'TEMP', 'TEMP_ATTRIBUTES', 'DEWP', 'DEWP_ATTRIBUTES', 'SLP', 'SLP_ATTRIBUTES', 'STP', 'STP_ATTRIBUTES', 'VISIB', 'VISIB_ATTRIBUTES', 'WDSP', 'WDSP_ATTRIBUTES', 'MXSPD', 'GUST', 'MAX', 'MIN', 'PRCP', 'SNDP', 'FRSHTT']


[Stage 48:==================================================>(2262 + 11) / 2282]

+-------+------------------+-------------------+-----------------+------------------+-----------------+------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+-----------------+-----------------+
|summary|LATITUDE          |LONGITUDE          |ELEVATION        |TEMP              |TEMP_ATTRIBUTES  |DEWP              |DEWP_ATTRIBUTES  |SLP              |SLP_ATTRIBUTES    |STP              |STP_ATTRIBUTES    |VISIB             |VISIB_ATTRIBUTES  |WDSP             |WDSP_ATTRIBUTES   |MXSPD             |GUST              |MAX               |MIN              |PRCP              |SNDP             |FRSHTT           |
+-------+------------------+-------------------+-----------------+------------------+-----------------+------------------+-----------------+-----------------+--

In [12]:
# ============================================================
# 9. Conteo de valores nulos reales por columna
# ============================================================

def missing_condition(column_name, data_type):
    c = F.col(f"`{column_name}`")
    condition = c.isNull()

    dtype = data_type.simpleString()
    if dtype == "string":
        normalized = F.lower(F.trim(c))
        condition = condition | (F.trim(c) == "") | normalized.isin(
            "na", "n/a", "nan", "null", "none", "missing"
        )
    elif dtype in {"float", "double"}:
        condition = condition | F.isnan(c)

    return condition

missing_exprs = [
    F.sum(F.when(missing_condition(field.name, field.dataType), 1).otherwise(0)).alias(field.name)
    for field in df.schema.fields
]

missing_counts = df.select(missing_exprs).collect()[0].asDict()

missing_rows = [
    (
        col,
        int(count),
        round((int(count) / num_records) * 100, 4) if num_records else 0.0
    )
    for col, count in missing_counts.items()
]

missing_df = spark.createDataFrame(
    missing_rows,
    ["columna", "valores_faltantes", "porcentaje_faltante"]
).orderBy(F.desc("valores_faltantes"))

missing_df.show(len(df.columns), truncate=False)

columns_with_missing = missing_df.filter(F.col("valores_faltantes") > 0).count()
print(f"Columnas con al menos un valor faltante: {columns_with_missing:,}")

[Stage 51:================================================>  (2155 + 11) / 2282]

+----------------+-----------------+-------------------+
|columna         |valores_faltantes|porcentaje_faltante|
+----------------+-----------------+-------------------+
|MAX_ATTRIBUTES  |13776738         |60.8542            |
|MIN_ATTRIBUTES  |13683130         |60.4407            |
|PRCP_ATTRIBUTES |1790385          |7.9084             |
|ELEVATION       |73465            |0.3245             |
|LATITUDE        |71491            |0.3158             |
|LONGITUDE       |71491            |0.3158             |
|NAME            |71489            |0.3158             |
|STP             |0                |0.0                |
|TEMP            |0                |0.0                |
|STATION         |0                |0.0                |
|TEMP_ATTRIBUTES |0                |0.0                |
|STP_ATTRIBUTES  |0                |0.0                |
|MAX             |0                |0.0                |
|VISIB           |0                |0.0                |
|DATE            |0            

In [13]:
# ============================================================
# 10. Identificación de valores centinela de NOAA
# ============================================================
# NOAA GSOD usa valores como 9999.9, 999.9 y 99.99 para representar datos faltantes.
# Estos valores NO siempre aparecen como null, por eso deben contarse aparte.

sentinel_values = {
    "TEMP": 9999.9,
    "DEWP": 9999.9,
    "SLP": 9999.9,
    "STP": 999.9,
    "VISIB": 999.9,
    "WDSP": 999.9,
    "MXSPD": 999.9,
    "GUST": 999.9,
    "MAX": 9999.9,
    "MIN": 9999.9,
    "PRCP": 99.99,
    "SNDP": 999.9,
    "ELEVATION": -999.9,
}

sentinel_rows = []
for column, sentinel in sentinel_values.items():
    if column in df.columns:
        count = df.filter(F.col(column) == F.lit(sentinel)).count()
        pct = round((count / num_records) * 100, 4) if num_records else 0.0
        sentinel_rows.append((column, float(sentinel), int(count), pct))

if sentinel_rows:
    sentinel_df = spark.createDataFrame(
        sentinel_rows,
        ["columna", "valor_centinela", "conteo", "porcentaje"]
    ).orderBy(F.desc("conteo"))
    sentinel_df.show(truncate=False)
else:
    print("No se encontraron columnas esperadas para revisar valores centinela.")

[Stage 94:================================================>  (2179 + 11) / 2282]

+---------+---------------+--------+----------+
|columna  |valor_centinela|conteo  |porcentaje|
+---------+---------------+--------+----------+
|SNDP     |999.9          |21089389|93.1554   |
|GUST     |999.9          |16306409|72.0282   |
|SLP      |9999.9         |7802110 |34.4632   |
|STP      |999.9          |6652873 |29.3869   |
|VISIB    |999.9          |5983089 |26.4283   |
|PRCP     |99.99          |1790385 |7.9084    |
|MXSPD    |999.9          |1107184 |4.8906    |
|DEWP     |9999.9         |1060786 |4.6857    |
|WDSP     |999.9          |838659  |3.7045    |
|MIN      |9999.9         |17706   |0.0782    |
|MAX      |9999.9         |14669   |0.0648    |
|ELEVATION|-999.9         |12604   |0.0557    |
|TEMP     |9999.9         |0       |0.0       |
+---------+---------------+--------+----------+



In [14]:
# ============================================================
# 11. Reemplazar valores centinela por NULL para análisis correcto
# ============================================================

df_clean = df

for column, sentinel in sentinel_values.items():
    if column in df_clean.columns:
        df_clean = df_clean.withColumn(
            column,
            F.when(F.col(column) == F.lit(sentinel), F.lit(None)).otherwise(F.col(column))
        )

df_clean = df_clean.cache()

print("Valores centinela reemplazados por NULL en df_clean.")
df_clean.select([c for c in ["TEMP", "PRCP", "DEWP", "SLP", "WDSP", "GUST", "SNDP"] if c in df_clean.columns]).describe().show(truncate=False)

Valores centinela reemplazados por NULL en df_clean.


[Stage 99:================================================>  (2167 + 12) / 2282]

+-------+------------------+-------------------+-----------------+------------------+-----------------+-----------------+-----------------+
|summary|TEMP              |PRCP               |DEWP             |SLP               |WDSP             |GUST             |SNDP             |
+-------+------------------+-------------------+-----------------+------------------+-----------------+-----------------+-----------------+
|count  |22638932          |20848547           |21578146         |14836822          |21800273         |6332523          |1549543          |
|mean   |56.034811010519356|0.07358172682249671|44.5882666332872 |1014.3586953324623|6.2519730005216  |22.55553547930269|9.837997267581475|
|stddev |22.864787194589624|0.3026127110631408 |21.64003259212112|9.180988413944021 |4.396459149560645|7.965420803520725|10.84596380107262|
|min    |-114.3            |0.0                |-122.7           |904.4             |0.0              |9.7              |0.4              |
|max    |110.0      

In [15]:
# ============================================================
# 12. Estadística climática relevante para el tema
# ============================================================

selected_cols = [c for c in ["TEMP", "MAX", "MIN", "PRCP", "WDSP", "GUST", "SLP", "VISIB"] if c in df_clean.columns]

summary_exprs = []
for c in selected_cols:
    summary_exprs.extend([
        F.count(c).alias(f"{c}_count"),
        F.round(F.mean(c), 4).alias(f"{c}_mean"),
        F.round(F.stddev(c), 4).alias(f"{c}_stddev"),
        F.min(c).alias(f"{c}_min"),
        F.max(c).alias(f"{c}_max"),
    ])

if summary_exprs:
    climate_summary = df_clean.select(summary_exprs)
    climate_summary.show(truncate=False)
else:
    print("No se encontraron columnas climáticas principales.")

[Stage 102:===============================================>  (2169 + 11) / 2282]

+----------+---------+-----------+--------+--------+---------+--------+----------+-------+-------+---------+--------+----------+-------+-------+----------+---------+-----------+--------+--------+----------+---------+-----------+--------+--------+----------+---------+-----------+--------+--------+---------+---------+----------+-------+-------+-----------+----------+------------+---------+---------+
|TEMP_count|TEMP_mean|TEMP_stddev|TEMP_min|TEMP_max|MAX_count|MAX_mean|MAX_stddev|MAX_min|MAX_max|MIN_count|MIN_mean|MIN_stddev|MIN_min|MIN_max|PRCP_count|PRCP_mean|PRCP_stddev|PRCP_min|PRCP_max|WDSP_count|WDSP_mean|WDSP_stddev|WDSP_min|WDSP_max|GUST_count|GUST_mean|GUST_stddev|GUST_min|GUST_max|SLP_count|SLP_mean |SLP_stddev|SLP_min|SLP_max|VISIB_count|VISIB_mean|VISIB_stddev|VISIB_min|VISIB_max|
+----------+---------+-----------+--------+--------+---------+--------+----------+-------+-------+---------+--------+----------+-------+-------+----------+---------+-----------+--------+--------+---

In [16]:
# ============================================================
# 13. Tendencia anual de temperatura y precipitación
# ============================================================

if "DATE_PARSED" not in df_clean.columns and "DATE" in df_clean.columns:
    df_clean = df_clean.withColumn("DATE_PARSED", F.to_date(F.col("DATE").cast("string")))

if "DATE_PARSED" in df_clean.columns:
    df_year = df_clean.withColumn("YEAR", F.year("DATE_PARSED"))

    agg_exprs = [F.count("*").alias("registros")]
    if "TEMP" in df_year.columns:
        agg_exprs.append(F.round(F.avg("TEMP"), 4).alias("temperatura_promedio_F"))
    if "PRCP" in df_year.columns:
        agg_exprs.append(F.round(F.avg("PRCP"), 4).alias("precipitacion_promedio_pulgadas"))
    if "STATION" in df_year.columns:
        agg_exprs.append(F.countDistinct("STATION").alias("estaciones_unicas"))

    yearly_summary = df_year.groupBy("YEAR").agg(*agg_exprs).orderBy("YEAR")
    yearly_summary.show(100, truncate=False)
else:
    print("No se encontró columna DATE para calcular tendencia anual.")

[Stage 105:=================================================>(2265 + 11) / 2282]

+----+---------+----------------------+-------------------------------+-----------------+
|YEAR|registros|temperatura_promedio_F|precipitacion_promedio_pulgadas|estaciones_unicas|
+----+---------+----------------------+-------------------------------+-----------------+
|2020|4119205  |55.5515               |0.0753                         |12299            |
|2021|4026550  |54.988                |0.0761                         |12275            |
|2022|3994699  |55.3889               |0.0721                         |12319            |
|2023|4038747  |56.4892               |0.0707                         |12311            |
|2024|3931419  |57.028                |0.0744                         |12159            |
|2025|2528312  |57.2396               |0.0725                         |11656            |
+----+---------+----------------------+-------------------------------+-----------------+



In [18]:
# ============================================================
# 14. Cobertura geográfica y sesgo de estaciones
# ============================================================

lat_col = "LATITUDE" if "LATITUDE" in df_clean.columns else None
lon_col = "LONGITUDE" if "LONGITUDE" in df_clean.columns else None

# Algunos archivos de metadata usan LAT/LON; los registros diarios suelen usar LATITUDE/LONGITUDE.
if lat_col is None and "LAT" in df_clean.columns:
    lat_col = "LAT"
if lon_col is None and "LON" in df_clean.columns:
    lon_col = "LON"

if lat_col and lon_col:
    geo_summary = df_clean.select(
        F.min(lat_col).alias("lat_min"),
        F.max(lat_col).alias("lat_max"),
        F.min(lon_col).alias("lon_min"),
        F.max(lon_col).alias("lon_max"),
        F.countDistinct(lat_col, lon_col).alias("ubicaciones_unicas")
    )
    geo_summary.show(truncate=False)

    if station_col:
        stations_geo = df_clean.select(station_col, lat_col, lon_col).dropDuplicates([station_col])
        stations_geo = stations_geo.withColumn(
            "region",
            F.when((F.col(lat_col) > 15) & (F.col(lon_col) < -30), "América del Norte")
             .when((F.col(lat_col) <= 15) & (F.col(lat_col) >= -60) & (F.col(lon_col) < -30), "América del Sur")
             .when((F.col(lat_col) > 35) & (F.col(lon_col) >= -30) & (F.col(lon_col) <= 60), "Europa")
             .when((F.col(lat_col) <= 35) & (F.col(lat_col) >= -40) & (F.col(lon_col) >= -20) & (F.col(lon_col) <= 60), "África")
             .when((F.col(lon_col) > 60) & (F.col(lat_col) > 0), "Asia")
             .when((F.col(lon_col) > 100) & (F.col(lat_col) < 0), "Oceanía")
             .otherwise("Otra")
        )

        from pyspark.sql.window import Window
        region_summary = (
            stations_geo.groupBy("region")
            .agg(F.count("*").alias("estaciones"))
            .withColumn("porcentaje", F.round(F.col("estaciones") / F.sum("estaciones").over(Window.partitionBy()) * 100, 2))
            .orderBy(F.desc("estaciones"))
        )

        region_summary = (
            stations_geo.groupBy("region")
            .agg(F.count("*").alias("estaciones"))
        )
        total_stations = region_summary.agg(F.sum("estaciones")).collect()[0][0]
        region_summary = region_summary.withColumn(
            "porcentaje",
            F.round(F.col("estaciones") / F.lit(total_stations) * 100, 2)
        ).orderBy(F.desc("estaciones"))

        region_summary.show(truncate=False)
else:
    print("No se encontraron columnas de latitud/longitud.")

+-------+-------+------------+-------+------------------+
|lat_min|lat_max|lon_min     |lon_max|ubicaciones_unicas|
+-------+-------+------------+-------+------------------+
|-90.0  |83.65  |-179.9833333|179.75 |13065             |
+-------+-------+------------+-------+------------------+



[Stage 133:=================================================>(2240 + 11) / 2282]

+-----------------+----------+----------+
|region           |estaciones|porcentaje|
+-----------------+----------+----------+
|América del Norte|3991      |30.68     |
|Europa           |3544      |27.24     |
|Asia             |2392      |18.39     |
|África           |1163      |8.94      |
|América del Sur  |1050      |8.07      |
|Oceanía          |749       |5.76      |
|Otra             |119       |0.91      |
+-----------------+----------+----------+

